# Chapter 7 — Qualitative Regressors (Dummy Variables)
## University of Neuchâtel — Python for Econometrics

Qualitative (categorical) variables must be encoded as **dummy variables** before
entering a regression. This chapter covers:

1. Creating dummy variables manually
2. Intercept shifts (baseline vs. non-baseline group)
3. Slope interactions (dummy × continuous)
4. Multiple categories (avoid the dummy variable trap)
5. Linear Probability Model (LPM)
6. Differences-in-Differences (DiD)

**Format:** Every step is in its own cell. We compute everything **manually** first,
then verify with `statsmodels`, and use `np.isclose()` to confirm equality.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Load Dataset: `wage1`

We use Wooldridge's **wage1** dataset.
Key variables:
- `wage`: hourly wage (USD)
- `educ`: years of education
- `exper`: years of experience
- `female`: 1 = female, 0 = male  ← **qualitative / dummy**
- `married`: 1 = married, 0 = single  ← **qualitative / dummy**
- `smsa`: 1 = lives in SMSA (metro area)

In [ ]:
df = woo.dataWoo('wage1')
print('Shape:', df.shape)

In [ ]:
print(df[['wage','educ','exper','female','married','smsa']].head(10))

In [ ]:
print('\nDescriptive statistics:')
print(df[['wage','educ','exper','female','married']].describe().round(3))

In [ ]:
print('\nShare female:', df['female'].mean().round(3))
print('Share married:', df['married'].mean().round(3))

## 2 — Intercept Shift: Gender Wage Gap

**Model:**

$$\text{wage}_i = \beta_0 + \delta_0 \cdot \text{female}_i + \beta_1 \cdot \text{educ}_i + \beta_2 \cdot \text{exper}_i + u_i$$

**Why use a dummy variable?** `female` is 0/1, so:
- Males ($\text{female}=0$): $E[\text{wage}] = \beta_0 + \beta_1 \text{educ} + \beta_2 \text{exper}$
- Females ($\text{female}=1$): $E[\text{wage}] = (\beta_0+\delta_0) + \beta_1 \text{educ} + \beta_2 \text{exper}$

$\delta_0$ measures the **wage differential** holding education and experience constant.

**When to use:** Any binary categorical variable (gender, treatment, region, etc.).

### Step 2.1 — Verify the dummy variable values

In [ ]:
print('Unique values of female:', df['female'].unique())
print('Count female=1:', (df['female']==1).sum())
print('Count female=0:', (df['female']==0).sum())

### Step 2.2 — Build design matrix manually (including dummy)

In [ ]:
n = len(df)
X = np.column_stack([
    np.ones(n),
    df['female'].values,
    df['educ'].values,
    df['exper'].values
])
y = df['wage'].values
print('X shape:', X.shape)
print('First 5 rows of X (const, female, educ, exper):')
print(X[:5])

### Step 2.3 — Manual OLS: $\hat{\beta} = (X'X)^{-1}X'y$

In [ ]:
XtX = X.T @ X
Xty = X.T @ y
beta_manual = np.linalg.solve(XtX, Xty)
print('Manual OLS coefficients:')
labels = ['const','female','educ','exper']
for lbl, b in zip(labels, beta_manual):
    print(f'  {lbl:10s}: {b:.4f}')

### Step 2.4 — Statsmodels OLS

In [ ]:
res_intercept = smf.ols('wage ~ female + educ + exper', data=df).fit()
print(res_intercept.summary())

### Step 2.5 — Compare manual vs statsmodels

In [ ]:
sm_params = res_intercept.params
print('Coefficient comparison:')
for lbl, b_man in zip(labels, beta_manual):
    b_sm = sm_params[lbl]
    match = np.isclose(b_man, b_sm)
    print(f'  {lbl:10s}: manual={b_man:.4f}, statsmodels={b_sm:.4f}, match={match}')

### Step 2.6 — Interpretation

In [ ]:
delta0 = res_intercept.params['female']
print(f'Gender wage gap (controlling for educ & exper): ${delta0:.2f}/hour')
print(f'Females earn on average ${abs(delta0):.2f} less per hour than males')
print(f'  (holding education and experience constant)')

## 3 — Slope Interaction: Does Education Return Differ by Gender?

**Model with interaction:**

$$\text{wage}_i = \beta_0 + \delta_0 \cdot \text{female}_i + \beta_1 \cdot \text{educ}_i + \delta_1 \cdot (\text{female}_i \times \text{educ}_i) + \beta_2 \text{exper}_i + u_i$$

- $\beta_1$: return to education for **males**
- $\beta_1 + \delta_1$: return to education for **females**
- $\delta_1$: **differential** return to education for females

**When to use:** When you suspect the slope (not just intercept) differs across groups.

### Step 3.1 — Create interaction term manually

In [ ]:
df['female_educ'] = df['female'] * df['educ']
print('Dataset with interaction term (first 8 rows):')
print(df[['female','educ','female_educ','wage']].head(8))

In [ ]:
print('Interaction = female * educ. Check:')
print('  female=1, educ=12 → female_educ =', 1*12)
print('  female=0, educ=16 → female_educ =', 0*16)

### Step 3.2 — Manual OLS with interaction

In [ ]:
X_int = np.column_stack([
    np.ones(n),
    df['female'].values,
    df['educ'].values,
    df['female_educ'].values,
    df['exper'].values
])
beta_int_manual = np.linalg.solve(X_int.T @ X_int, X_int.T @ y)
labels_int = ['const','female','educ','female_educ','exper']
print('Manual OLS with interaction:')
for lbl, b in zip(labels_int, beta_int_manual):
    print(f'  {lbl:15s}: {b:.4f}')

### Step 3.3 — Statsmodels with interaction

In [ ]:
res_interact = smf.ols('wage ~ female + educ + female_educ + exper', data=df).fit()
print(res_interact.summary())

### Step 3.4 — Compare manual vs statsmodels (interaction model)

In [ ]:
sm_int = res_interact.params
print('Coefficient comparison (interaction model):')
for lbl, b_man in zip(labels_int, beta_int_manual):
    b_sm = sm_int[lbl]
    match = np.isclose(b_man, b_sm)
    print(f'  {lbl:15s}: manual={b_man:.4f}, sm={b_sm:.4f}, match={match}')

### Step 3.5 — Interpret differential returns

In [ ]:
b1     = res_interact.params['educ']
d1     = res_interact.params['female_educ']
print(f'Return to education (males):   ${b1:.3f}/hour per year')
print(f'Return to education (females): ${b1+d1:.3f}/hour per year')
print(f'Differential (delta_1):        ${d1:.3f}/hour per year')
print(f'Is delta_1 statistically significant? p={res_interact.pvalues["female_educ"]:.4f}')

### Step 3.6 — Visualise intercept + slope interaction

In [ ]:
educ_range = np.linspace(df['educ'].min(), df['educ'].max(), 100)
exper_mean = df['exper'].mean()
b = res_interact.params
wage_male   = b['Intercept'] + b['educ']*educ_range + b['exper']*exper_mean
wage_female = (b['Intercept']+b['female']) + (b['educ']+b['female_educ'])*educ_range + b['exper']*exper_mean

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(educ_range, wage_male,   label='Male',   linewidth=2)
ax.plot(educ_range, wage_female, label='Female', linewidth=2, linestyle='--')
ax.set_xlabel('Years of Education')
ax.set_ylabel('Predicted Wage ($/hr)')
ax.set_title('Wage–Education Lines by Gender (Interaction Model)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch07_01_gender_wage_lines.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4 — Multiple Categories: Occupation Dummies

When a categorical variable has $k$ categories, we need **$k-1$ dummies**
(one is the **baseline / reference category** — omitted to avoid perfect collinearity).

**Dummy Variable Trap:** Including all $k$ dummies causes perfect multicollinearity
(they sum to 1 = the constant column).

**Rule:** Always omit one category (the baseline). All coefficients are interpreted
relative to the baseline.

### Step 4.1 — Load data and inspect occupation variable

In [ ]:
df2 = woo.dataWoo('wage1')
print('Occupation columns available:')
occ_cols = [c for c in df2.columns if 'occ' in c or 'serv' in c or 'trade' in c]
print(occ_cols)
print(df2[['profocc','clerocc','servocc']].describe())

In [ ]:
# profocc=1: professional/managerial, clerocc=1: clerical, servocc=1: service
# baseline: other occupations (all three = 0)
print('Occupation counts:')
print('Professional:', df2['profocc'].sum())
print('Clerical:    ', df2['clerocc'].sum())
print('Service:     ', df2['servocc'].sum())
print('Other (baseline):', (df2[['profocc','clerocc','servocc']].sum(axis=1)==0).sum())

### Step 4.2 — Manual OLS with occupation dummies

In [ ]:
n2 = len(df2)
X_occ = np.column_stack([
    np.ones(n2),
    df2['educ'].values,
    df2['exper'].values,
    df2['female'].values,
    df2['profocc'].values,
    df2['clerocc'].values,
    df2['servocc'].values
])
y2 = df2['wage'].values
beta_occ = np.linalg.solve(X_occ.T @ X_occ, X_occ.T @ y2)
labels_occ = ['const','educ','exper','female','profocc','clerocc','servocc']
print('Manual OLS with occupation dummies:')
for lbl, b in zip(labels_occ, beta_occ):
    print(f'  {lbl:10s}: {b:.4f}')

### Step 4.3 — Statsmodels OLS with occupation dummies

In [ ]:
res_occ = smf.ols('wage ~ educ + exper + female + profocc + clerocc + servocc', data=df2).fit()
print(res_occ.summary())

### Step 4.4 — Compare manual vs statsmodels (occupation model)

In [ ]:
sm_occ = res_occ.params
print('Coefficient comparison (occupation model):')
for lbl, b_man in zip(labels_occ, beta_occ):
    b_sm = sm_occ[lbl]
    match = np.isclose(b_man, b_sm)
    print(f'  {lbl:10s}: manual={b_man:.4f}, sm={b_sm:.4f}, match={match}')

### Step 4.5 — Interpret occupation effects

In [ ]:
print('Wage premium relative to baseline (other) occupation:')
for occ in ['profocc','clerocc','servocc']:
    coef = res_occ.params[occ]
    pval = res_occ.pvalues[occ]
    sig  = '***' if pval<0.01 else ('**' if pval<0.05 else ('*' if pval<0.1 else ''))
    print(f'  {occ}: ${coef:+.2f}/hr  (p={pval:.3f}) {sig}')

## 5 — Linear Probability Model (LPM)

**When the dependent variable is binary (0/1)**, OLS is called the
**Linear Probability Model**:

$$P(y=1 | x) = \beta_0 + \beta_1 x_1 + \cdots + \beta_k x_k$$

**Interpretation:** $\beta_j$ = change in **probability** of $y=1$ for a unit increase in $x_j$.

**When to use:** Quick approximation for binary outcomes. Works well near the middle
of the data. Limitations: predicted probabilities can exceed [0,1]; heteroskedastic errors.

**Dataset:** `mroz` — female labour force participation.

### Step 5.1 — Load mroz data

In [ ]:
mroz = woo.dataWoo('mroz')
print('Shape:', mroz.shape)
print(mroz[['inlf','educ','exper','kidslt6','nwifeinc']].head(8))

In [ ]:
print('\nDependent variable inlf (in labour force):')
print('  inlf=1:', mroz['inlf'].sum(), '  inlf=0:', (mroz['inlf']==0).sum())
print('  Proportion participating:', mroz['inlf'].mean().round(3))

### Step 5.2 — Manual LPM: $\hat{\beta} = (X'X)^{-1}X'y$

In [ ]:
n_m = len(mroz)
X_lpm = np.column_stack([
    np.ones(n_m),
    mroz['educ'].values,
    mroz['exper'].values,
    mroz['kidslt6'].values,
    mroz['nwifeinc'].values
])
y_lpm = mroz['inlf'].values.astype(float)
beta_lpm = np.linalg.solve(X_lpm.T @ X_lpm, X_lpm.T @ y_lpm)
labels_lpm = ['const','educ','exper','kidslt6','nwifeinc']
print('Manual LPM coefficients:')
for lbl, b in zip(labels_lpm, beta_lpm):
    print(f'  {lbl:12s}: {b:.5f}')

### Step 5.3 — Statsmodels LPM

In [ ]:
res_lpm = smf.ols('inlf ~ educ + exper + kidslt6 + nwifeinc', data=mroz).fit()
print(res_lpm.summary())

### Step 5.4 — Compare manual vs statsmodels (LPM)

In [ ]:
sm_lpm = res_lpm.params
print('LPM coefficient comparison:')
for lbl, b_man in zip(labels_lpm, beta_lpm):
    b_sm = sm_lpm[lbl]
    match = np.isclose(b_man, b_sm)
    print(f'  {lbl:12s}: manual={b_man:.5f}, sm={b_sm:.5f}, match={match}')

### Step 5.5 — Check for out-of-range predicted probabilities

In [ ]:
y_hat = res_lpm.fittedvalues
print('Predicted probabilities summary:')
print(f'  Min: {y_hat.min():.3f}')
print(f'  Max: {y_hat.max():.3f}')
n_below = (y_hat < 0).sum()
n_above = (y_hat > 1).sum()
print(f'  Below 0: {n_below} ({100*n_below/len(y_hat):.1f}%)')
print(f'  Above 1: {n_above} ({100*n_above/len(y_hat):.1f}%)')
print('\nNote: Out-of-range predictions are a known LPM limitation.')

### Step 5.6 — LPM interpretation

In [ ]:
print('LPM coefficient interpretation (effect on P(inlf=1)):')
for lbl in ['educ','exper','kidslt6','nwifeinc']:
    b = res_lpm.params[lbl]
    p = res_lpm.pvalues[lbl]
    print(f'  {lbl:12s}: {b:+.4f} ({b*100:+.2f} pp per unit)  p={p:.3f}')

## 6 — Differences-in-Differences (DiD)

**Question:** Does a policy/treatment affect outcomes?

**Setup:**
- Two groups: **treated** ($d=1$) and **control** ($d=0$)
- Two periods: **before** ($t=0$) and **after** ($t=1$) treatment

**Model:**

$$y_{it} = \beta_0 + \beta_1 d_i + \beta_2 t + \delta \cdot (d_i \times t) + u_{it}$$

| | Before ($t=0$) | After ($t=1$) | Difference |
|---|---|---|---|
| Control | $\beta_0$ | $\beta_0+\beta_2$ | $\beta_2$ |
| Treated | $\beta_0+\beta_1$ | $\beta_0+\beta_1+\beta_2+\delta$ | $\beta_2+\delta$ |
| **DiD** | | | $\delta$ |

**Key assumption:** Parallel trends — in the absence of treatment, the two groups
would have followed the same trend.

**When to use:** Natural experiments, policy evaluation (min wage, training programs, etc.).

### Step 6.1 — Simulate DiD data manually

In [ ]:
np.random.seed(42)
n_did = 200
# half treated, half control; half before, half after
d = np.repeat([0,0,1,1], n_did//4)
t = np.tile([0,1], n_did//2)
true_delta = 2.5  # true treatment effect
y_did = 5 + 1.5*d + 3*t + true_delta*(d*t) + np.random.normal(0, 1, n_did)
df_did = pd.DataFrame({'y': y_did, 'd': d, 't': t, 'dt': d*t})
print('DiD dataset (first 8 rows):')
print(df_did.head(8))
print(f'\nTrue treatment effect (delta): {true_delta}')

### Step 6.2 — Compute DiD manually: group means

In [ ]:
mean_c0 = df_did.loc[(df_did['d']==0)&(df_did['t']==0), 'y'].mean()
mean_c1 = df_did.loc[(df_did['d']==0)&(df_did['t']==1), 'y'].mean()
mean_t0 = df_did.loc[(df_did['d']==1)&(df_did['t']==0), 'y'].mean()
mean_t1 = df_did.loc[(df_did['d']==1)&(df_did['t']==1), 'y'].mean()
print(f'Control before:  {mean_c0:.3f}')
print(f'Control after:   {mean_c1:.3f}')
print(f'Treated before:  {mean_t0:.3f}')
print(f'Treated after:   {mean_t1:.3f}')

In [ ]:
did_manual = (mean_t1 - mean_t0) - (mean_c1 - mean_c0)
print(f'\nDiD manual = (T_after - T_before) - (C_after - C_before)')
print(f'           = ({mean_t1:.3f} - {mean_t0:.3f}) - ({mean_c1:.3f} - {mean_c0:.3f})')
print(f'           = {did_manual:.3f}  (true: {true_delta})')

### Step 6.3 — OLS regression: $y = \beta_0 + \beta_1 d + \beta_2 t + \delta (d \times t) + u$

In [ ]:
n_d = len(df_did)
X_did = np.column_stack([
    np.ones(n_d),
    df_did['d'].values,
    df_did['t'].values,
    df_did['dt'].values
])
y_d_arr = df_did['y'].values
beta_did = np.linalg.solve(X_did.T @ X_did, X_did.T @ y_d_arr)
labels_did = ['const','d','t','dt']
print('Manual OLS DiD coefficients:')
for lbl, b in zip(labels_did, beta_did):
    print(f'  {lbl:6s}: {b:.4f}')

### Step 6.4 — Statsmodels DiD

In [ ]:
res_did = smf.ols('y ~ d + t + dt', data=df_did).fit()
print(res_did.summary())

### Step 6.5 — Compare manual DiD: group-means vs OLS vs true

In [ ]:
delta_reg = res_did.params['dt']
match_did = np.isclose(did_manual, delta_reg)
print('DiD comparison:')
print(f'  Group-means DiD: {did_manual:.4f}')
print(f'  OLS (dt coeff):  {delta_reg:.4f}')
print(f'  True delta:      {true_delta:.4f}')
print(f'  group-means == OLS? {match_did}')

### Step 6.6 — Visualise parallel trends and DiD

In [ ]:
means = df_did.groupby(['d','t'])['y'].mean().unstack()
fig, ax = plt.subplots(figsize=(7,5))
ax.plot([0,1], [means.loc[0,0], means.loc[0,1]], 'bs-', label='Control', linewidth=2)
ax.plot([0,1], [means.loc[1,0], means.loc[1,1]], 'ro-', label='Treated', linewidth=2)
counterfactual_after = means.loc[1,0] + (means.loc[0,1]-means.loc[0,0])
ax.plot([1], [counterfactual_after], 'r^', markersize=10, label='Treated counterfactual', alpha=0.6)
ax.annotate('', xy=(1.02, means.loc[1,1]), xytext=(1.02, counterfactual_after),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(1.05, (means.loc[1,1]+counterfactual_after)/2, f'DiD={did_manual:.2f}', color='green', va='center')
ax.set_xticks([0,1])
ax.set_xticklabels(['Before','After'])
ax.set_ylabel('Mean y')
ax.set_title('Differences-in-Differences')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch07_02_did.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7 — Summary: Dummy Variable Results

| Model | Key Coefficient | Value | Interpretation |
|---|---|---|---|
| Intercept shift | `female` | ~ −1.81 | Women earn ~$1.81/hr less |
| Slope interaction | `female_educ` | varies | Differential return to educ |
| Occupation | `profocc` | varies | Premium over baseline |
| LPM | `educ` | ~ +0.038 | +1 yr educ → +3.8pp participation |
| DiD | `dt` | ~ 2.5 | Treatment effect = +2.5 units |

In [ ]:
summary_data = {
    'Model': ['Intercept shift','Slope interaction','LPM (educ)','DiD'],
    'Coefficient': [
        res_intercept.params['female'],
        res_interact.params['female_educ'],
        res_lpm.params['educ'],
        res_did.params['dt']
    ],
    'p-value': [
        res_intercept.pvalues['female'],
        res_interact.pvalues['female_educ'],
        res_lpm.pvalues['educ'],
        res_did.pvalues['dt']
    ]
}
df_summary = pd.DataFrame(summary_data)
df_summary['Coefficient'] = df_summary['Coefficient'].round(4)
df_summary['p-value'] = df_summary['p-value'].round(4)
print(df_summary.to_string(index=False))
df_summary.to_csv(os.path.join(output_dir,'table_ch07_summary.csv'), index=False)
print('\nSaved to output/table_ch07_summary.csv')